===========================================================

Projet de Machine Learning - Prédiction du MVP NBA 2026 v1

===========================================================

In [ ]:
# Import des DataSets et Création du DataFrame 

import pandas as pd
# Importation des fichiers à traiter
# Si projet sur kaggle : "/kaggle/input/datasets/sumitrodatta/nba-aba-baa-stats/fichier.csv"
df_stats             = pd.read_csv("Advanced.csv")
df_votes             = pd.read_csv("Player Award Shares.csv")
df_team_summaries        = pd.read_csv("Team Summaries.csv")
df_totals_team       = pd.read_csv("Team Totals.csv")
df_stats_per_games_p = pd.read_csv("Player Per Game.csv")

# Filtrage des données 
df_votes = df_votes[df_votes["award"] == "nba mvp"].copy()
df_stats = df_stats[df_stats["lg"] == "NBA"].copy()
df_stats = df_stats[df_stats["g"] >= 40].copy()
df_stats = df_stats[df_stats["mp"] >= 1000].copy()
df_stats_per_games_p = df_stats_per_games_p[df_stats_per_games_p["lg"] == "NBA"].copy()

# Création des moyennes de stats des équipes par match
df_team = df_totals_team[
    (df_totals_team["lg"] == "NBA") & (df_totals_team["playoffs"] == False)
].copy()

df_team["pts_per_game_t"] = df_team["pts"] / df_team["g"]
df_team["ast_per_game_t"] = df_team["ast"] / df_team["g"]
df_team["trb_per_game_t"] = df_team["trb"] / df_team["g"]
df_team["stl_per_game_t"] = df_team["stl"] / df_team["g"]
df_team["blk_per_game_t"] = df_team["blk"] / df_team["g"]

# Récupération du bilan d'équipe depuis Team Summaries
df_bilan = df_team_summaries[(df_team_summaries["lg"] == "NBA") & (df_team_summaries["playoffs"] == False)
].copy()
df_bilan["win_pct"] = df_bilan["w"] / (df_bilan["w"] + df_bilan["l"])
df_bilan = df_bilan[["season", "abbreviation", "win_pct"]].copy()

# Ajout du bilan dans df_team
df_team = pd.merge(
    df_team,
    df_bilan,
    on=["season", "abbreviation"],
    how="left"
)

# On garde uniquement ce dont on a besoin
df_team = df_team[[
    "season", "abbreviation",
    "pts_per_game_t", "ast_per_game_t",
    "trb_per_game_t", "stl_per_game_t", "blk_per_game_t",
    "win_pct"
]].copy()


# Fusion des tables joueurs et équipes pour que les joueurs soient liés à la bonne équipe
df_stats_per_games_p = pd.merge(
    df_stats_per_games_p,
    df_team,
    left_on=["season", "team"],        
    right_on=["season", "abbreviation"],
    how="left"
)

# Calcul de la contribution des joueurs à leur équipe
df_stats_per_games_p["pts_percent"] = (
    df_stats_per_games_p["pts_per_game"] /
    df_stats_per_games_p["pts_per_game_t"]
).round(4)

colonnes_stats_per_game = [
    "season", "player_id",        
    "pts_per_game", "ast_per_game",
    "trb_per_game", "stl_per_game",
    "blk_per_game", "tov_per_game",
    "pf_per_game",
    "pts_percent",
    "win_pct"
]

colonnes_stats_percent = [
    "season", "player", "player_id", "age", "team", "pos",
    "g", "mp",
    "per", "ts_percent",
    "ws", "ws_48",
    "bpm", "obpm", "dbpm", "vorp",
    "usg_percent", "ast_percent",
    "trb_percent", "blk_percent",
    "stl_percent", "tov_percent"
]
df_stats = df_stats[colonnes_stats_percent].copy()


colonnes_votes = ["season", "player_id", "player", "pts_won", "pts_max", "share", "winner"]
df_votes = df_votes[colonnes_votes].copy()
df_votes = df_votes.rename(columns={"share": "pct_votes", "winner": "est_mvp"})


# Stats avancées + Stats par match
df_final = pd.merge(
    df_stats,
    df_stats_per_games_p[colonnes_stats_per_game],
    on=["season", "player_id"],
    how="left"
)

# Résultats + votes MVP
df_final = pd.merge(
    df_final,
    df_votes[["season", "player_id", "pct_votes", "est_mvp"]],
    on=["season", "player_id"],
    how="left"
)


df_final["pct_votes"] = df_final["pct_votes"].fillna(0)
df_final["est_mvp"]   = (df_final["est_mvp"] == True).astype(int)
df_final = df_final[df_final["season"] >= 1980].copy()

df_final.to_csv("dataset_mvp_final.csv", index=False)
print("\ndataset_mvp_final.csv sauvegardé !")

In [ ]:
# Définition des catégories à observer
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")


df = pd.read_csv("dataset_mvp_final.csv")


# ==============================================================================
#
# Impact du joueur sur son équipe (45% du score final)
# 
# ==============================================================================

# VOTT 

df["vott"] = (
    df["usg_percent"] * 0.30 +   # Points/possessions : poids fort
    df["ast_percent"] * 0.20 +   # Passes décisives
    df["trb_percent"] * 0.20 +   # Rebonds totaux
    df["blk_percent"] * 0.15 +   # Contres
    df["stl_percent"] * 0.15 -   # Interceptions
    df["tov_percent"] * 0.05     # Pénalité pertes de balle
)

df["vott_norm"]  = df.groupby("season")["vott"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)
df["rang_vott"]  = df.groupby("season")["vott"].rank(ascending=False, method="min")
df["top3_vott"]  = (df["rang_vott"] <= 3).astype(int)

# VORP
df["vorp_norm"]  = df.groupby("season")["vorp"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)
df["rang_vorp"]  = df.groupby("season")["vorp"].rank(ascending=False, method="min")
df["top3_vorp"]  = (df["rang_vorp"] <= 3).astype(int)

# BPM 
df["bpm_norm"]   = df.groupby("season")["bpm"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)
df["rang_bpm"]   = df.groupby("season")["bpm"].rank(ascending=False, method="min")
df["rang_obpm"]  = df.groupby("season")["obpm"].rank(ascending=False, method="min")
df["rang_dbpm"]  = df.groupby("season")["dbpm"].rank(ascending=False, method="min")
df["top3_bpm"]   = (df["rang_bpm"] <= 3).astype(int)

# WS/48
df["ws48_norm"]  = df.groupby("season")["ws_48"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)
df["rang_ws48"]  = df.groupby("season")["ws_48"].rank(ascending=False, method="min")
df["top3_ws48"]  = (df["rang_ws48"] <= 3).astype(int)

# Création du score IMPACT représentant l'impact du joueur sur son équipe
df["score_impact"] = (
    df["vott_norm"]  * 0.40 +
    df["vorp_norm"]  * 0.25 +
    df["bpm_norm"]   * 0.20 +
    df["ws48_norm"]  * 0.15
)
df["rang_impact"] = df.groupby("season")["score_impact"].rank(ascending=False, method="min")


# ==============================================================================
#
# Impact du joueur sur la ligue (30% du score final)
# 
# ==============================================================================

# PER = résumé global pts/reb/ast/blk/stl normalisés
df["per_norm"]     = df.groupby("season")["per"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# TS% = efficacité tir (%tir, %3pts, %lancers-francs combinés)
df["ts_norm"]      = df.groupby("season")["ts_percent"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# Volume de tir = usg_percent (quantité de tentatives)
df["usg_norm"]     = df.groupby("season")["usg_percent"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# Défense avancée = stl_percent + blk_percent
df["def_avancee"]  = df["stl_percent"] + df["blk_percent"]
df["def_norm"]     = df.groupby("season")["def_avancee"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# Ratio ast/tov = qualité des passes (évite les pertes de balle)
df["ast_tov"]      = df["ast_percent"] / (df["tov_percent"] + 1e-9)
df["ast_tov_norm"] = df.groupby("season")["ast_tov"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

df["score_stats"] = (
    df["per_norm"]     * 0.35 +  # efficacité globale
    df["ts_norm"]      * 0.25 +  # efficacité au tir
    df["usg_norm"]     * 0.20 +  # volume de jeu
    df["def_norm"]     * 0.15 +  # impact défensif (blk + stl)
    df["ast_tov_norm"] * 0.05    # qualité des passes
)
df["rang_stats"] = df.groupby("season")["score_stats"].rank(ascending=False, method="min")


# ==============================================================================
#
# Bilan de l'équipe (25% du score final)
#
#
# ==============================================================================

# Projection sur 100 matchs
df["wins_sur_100"]   = (df["ws_48"] * 4 * 100).clip(upper=100)
df["wins100_norm"]   = df.groupby("season")["wins_sur_100"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

# Contribution réelle aux victoires
df["ws_norm_bilan"]  = df.groupby("season")["ws"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

df["score_bilan"] = (
    df["wins100_norm"]  * 0.60 +
    df["ws_norm_bilan"] * 0.40
)
df["rang_bilan"] = df.groupby("season")["score_bilan"].rank(ascending=False, method="min")


# ==============================================================================
#
# INDICE MVP FINAL
#
# ==============================================================================

df["indice_mvp"] = (
    df["score_impact"] * 0.40 +
    df["score_stats"]  * 0.35 +
    df["score_bilan"]  * 0.25
)
df["rang_indice_mvp"] = df.groupby("season")["indice_mvp"].rank(ascending=False, method="min")




In [ ]:
# =======================================================================================

# Machine Learning et Entrainement 

# =======================================================================================

FEATURES = [
    "vott", "vott_norm", "rang_vott", "top3_vott",
    "vorp", "vorp_norm", "rang_vorp", "top3_vorp",
    "bpm",  "obpm", "dbpm", "bpm_norm", "rang_bpm", "top3_bpm",
    "ws_48", "ws48_norm", "rang_ws48", "top3_ws48",
    "score_impact", "rang_impact",

    "per", "per_norm",
    "ts_percent", "ts_norm",
    "usg_percent", "usg_norm",
    "def_avancee", "def_norm",
    "ast_percent", "ast_tov", "ast_tov_norm",
    "blk_percent", "stl_percent",
    "score_stats", "rang_stats",

    "ws",
    "wins_sur_100", "wins100_norm",
    "score_bilan", "rang_bilan",


    "indice_mvp", "rang_indice_mvp",

    "age", "g"
]

df_model     = df[FEATURES + ["pct_votes", "season", "player"]].dropna()
saison_max   = df_model["season"].max()
train        = df_model[df_model["season"] < saison_max]
test         = df_model[df_model["season"] == saison_max].copy()
X = train[FEATURES]
y = train["pct_votes"]


model = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=4,
    random_state=1
)


model.fit(X, y)



In [ ]:
# =======================================================================================

#Prédiction finale pour le NBA MVP 2026

# =======================================================================================

test["predicted_share"] = model.predict(test[FEATURES]).clip(0)
test["predicted_pct"]   = (
    test["predicted_share"] / test["predicted_share"].sum() * 100
).round(1)

print(f"\nPrédiction MVP saison {saison_max} — Top 10 :")
print(
    test.sort_values("predicted_pct", ascending=False)
    .head(10)[["player", "predicted_pct", "indice_mvp",
               "score_impact", "score_stats", "score_bilan"]]
    .to_string(index=False)
)

test.to_csv("predictions_mvp_2026.csv", index=False)
print("\npredictions_mvp_2026.csv sauvegardé !")